# 08 - Gold KPI Dashboard

Observability notebook for the Gold layer. Visualizes all 5 KPIs with macro shock annotations.

**This notebook contains no processing logic** — all KPI computations happen in `src/medallion/gold.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from config import GOLD_DIR, MEDALLION_METADATA_DIR, MACRO_SHOCK_YEARS

## Load Gold manifest

In [ ]:
import json

manifest_path = MEDALLION_METADATA_DIR / "gold_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"Gold manifest written at: {manifest['written_at']}")
    for kpi, info in manifest["kpis"].items():
        print(f"  {kpi}: {info}")
    print(f"\nCaveats: {manifest['caveats']}")
else:
    print("No Gold manifest found. Run: make gold")

In [ ]:
SHOCK_LABELS = {1995: "Mortgage\nliberalization", 2008: "Lehman", 2020: "COVID", 2022: "Rate\nhikes"}

def add_shock_lines(ax, ymin=None, ymax=None):
    """Add vertical macro shock markers."""
    for year in MACRO_SHOCK_YEARS:
        ax.axvline(f"{year}-Q1", color="red", alpha=0.3, linestyle="--", linewidth=0.8)
        if ymax is not None:
            ax.text(f"{year}-Q1", ymax * 0.95, SHOCK_LABELS.get(year, str(year)),
                    fontsize=7, color="red", alpha=0.6, ha="center")

## KPI 1: Regional Price Index (1992=100)

In [ ]:
price_idx = pd.read_parquet(GOLD_DIR / "kpi_sqm_price_index.parquet")
regions = price_idx["region"].unique()

fig, ax = plt.subplots(figsize=(14, 5))
for region in regions:
    r = price_idx[price_idx["region"] == region].sort_values("quarter_id")
    ax.plot(r["quarter_id"], r["price_index"], label=region, linewidth=1.2)

ax.axhline(100, color="gray", linestyle=":", linewidth=0.8)
add_shock_lines(ax, ymax=price_idx["price_index"].max())
ax.set_title("Regional Real Price Index (1992=100)")
ax.set_ylabel("Price Index")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.xaxis.set_major_locator(plt.MaxNLocator(15))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"{len(price_idx)} rows, {len(regions)} regions")

## KPI 2: Transaction Volume

In [ ]:
volume = pd.read_parquet(GOLD_DIR / "kpi_transaction_volume.parquet")

vol_agg = volume.groupby("quarter_id")["n_transactions"].sum().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(vol_agg.index, vol_agg.values, width=0.8, color="steelblue", alpha=0.7)
add_shock_lines(ax, ymax=vol_agg.max())
ax.set_title("National Transaction Volume per Quarter")
ax.set_ylabel("Transactions")
ax.xaxis.set_major_locator(plt.MaxNLocator(15))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## KPI 3: Peak-to-Trough Drawdown

In [ ]:
drawdown = pd.read_parquet(GOLD_DIR / "kpi_drawdown.parquet")

if not drawdown.empty:
    print(f"{len(drawdown)} drawdown episodes detected\n")
    display_cols = ["region", "peak_quarter_id", "trough_quarter_id", 
                    "drawdown_pct", "drawdown_duration_quarters", "macro_shock_label"]
    display(drawdown[display_cols].sort_values("drawdown_pct"))

    fig, ax = plt.subplots(figsize=(10, 5))
    for _, row in drawdown.iterrows():
        color = "red" if row["macro_shock_label"] else "gray"
        label = row["macro_shock_label"] or ""
        ax.barh(
            f"{row['region']}\n{row['peak_quarter_id']}",
            row["drawdown_pct"],
            color=color, alpha=0.7,
        )
    ax.set_xlabel("Drawdown %")
    ax.set_title("Peak-to-Trough Drawdown Episodes")
    plt.tight_layout()
    plt.show()
else:
    print("No drawdown episodes found.")

## KPI 4: Inter-Quarterly Volatility

In [ ]:
volatility = pd.read_parquet(GOLD_DIR / "kpi_volatility.parquet")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for region in volatility["region"].unique():
    r = volatility[volatility["region"] == region].sort_values("quarter_id")
    axes[0].plot(r["quarter_id"], r["rolling_4q_std"], label=region, linewidth=1)
    axes[1].plot(r["quarter_id"], r["rolling_8q_std"], label=region, linewidth=1)

axes[0].set_title("Rolling 4Q Std Dev")
axes[1].set_title("Rolling 8Q Std Dev")
for ax in axes:
    ax.legend(fontsize=7, ncol=2)
    ax.xaxis.set_major_locator(plt.MaxNLocator(10))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

plt.suptitle("Inter-Quarterly Price Volatility", y=1.02)
plt.tight_layout()
plt.show()

## KPI 5: Volume-Bond Yield Elasticity

In [ ]:
elasticity = pd.read_parquet(GOLD_DIR / "kpi_bond_elasticity.parquet")

if not elasticity.empty:
    display(elasticity)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.errorbar(
        elasticity["lag_quarters"],
        elasticity["beta_ols"],
        yerr=elasticity["beta_ols_se"] * 1.96,
        fmt="o-", capsize=5, color="darkred",
    )
    ax.axhline(0, color="gray", linestyle=":", linewidth=0.8)
    ax.set_xlabel("Lag (quarters)")
    ax.set_ylabel("Beta (OLS)")
    ax.set_title("Volume-Bond Yield Elasticity (confounded OLS)")
    ax.set_xticks(elasticity["lag_quarters"])
    plt.tight_layout()
    plt.show()
else:
    print("No elasticity results. Needs statsmodels.")

## Summary

In [ ]:
print("Gold KPI Summary")
print("=" * 50)
print(f"Price Index:  {len(price_idx)} rows, {price_idx['region'].nunique()} regions")
print(f"Volume:       {len(volume)} rows, {volume['n_transactions'].sum():,} total transactions")
print(f"Drawdown:     {len(drawdown)} episodes")
print(f"Volatility:   {len(volatility)} rows")
print(f"Elasticity:   {len(elasticity)} lag configs")